# Push-T variable goal, run 3: goal-conditioned PPO expert → distilled BC teacher

v1 of the project (`CODEX_CONTEXT.md`) needs teachers that can push the T to **any goal position**, with the goal
in their observation. The transformer will later see demos but never the goal, and will train on Q1–Q3 goals only.
This notebook makes the full-state teachers:

1. **Variable-goal environment** `PushT-VariableGoal-v1` (`variable_goal_env.py`): PushT-v1 with the goal T's
   position random each episode, within ±10 cm of PushT-v1's original goal, in quadrants Q1 (+x,+y), Q2 (−x,+y),
   Q3 (−x,−y), Q4 (+x,−y). Goal rotation fixed. The success check is fixed to use each episode's own goal.
   **The stick tip also starts at a random spot** (±15 cm of the same centre, away from the T), placed by inverse
   kinematics. The movable T spawns as in PushT-v1. Starts are independent of the goal. **Teachers train on all four quadrants.**
2. **Goal-conditioned PPO expert**, trained here with ManiSkill's state PPO baseline (Jimmy's PPO can replace it later).
3. **BC teacher distilled from that expert:** noisy expert rollouts plus one training run. **No DAgger** (run 2 showed it was not needed).
4. **Per-quadrant validation and test** of both teachers, then both are saved under `teachers/`.

| Setting | Value |
|---|---|
| Observation / controller | `obs_mode="state"` (31 numbers, now including a varying `goal_pos`), `pd_joint_delta_pos` (7) |
| PPO | ManiSkill `ppo.py`: 1,024 envs, 16 steps per rollout, 32 minibatches, 8 epochs, γ = 0.99, 50M transitions, seed 1 |
| BC teacher | MLP 2 × 256 ReLU, tanh output, MSE, Adam 3e-4 cosine, batch 1024, 200k iterations |
| Simulator | ManiSkill v3.0.1 (commit `a4a4f92`), GPU physics |
| Seeds | expert data 100,000+; BC validation data 300,000+; closed-loop validation 1001; test 5001 (once, at the end) |

Select **Runtime → Change runtime type → L4 GPU** and run the cells in order. Upload only this notebook.
Outputs go to `MyDrive/182_final_project/state_bc/<session>/`. PPO is the long step (roughly 30–60 minutes, not yet measured).

In [ ]:
from pathlib import Path
import os, sys, json, re, subprocess, datetime, shlex, shutil

MANISKILL_COMMIT = "a4a4f9272ad64b1564035874b605ceb687b63ed8"  # v3.0.1, same as the other notebooks
MS_ROOT = Path("/content/ManiSkill")
WORK_ROOT = Path("/content/pusht_state_bc")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
USE_GOOGLE_DRIVE = True
DRIVE_SUBDIR = "182_final_project/state_bc"
RUN_LABEL = "run3_variable_goal"
ENV_ID = "PushT-VariableGoal-v1"

# PPO expert (ManiSkill's state baseline settings, as in Neil's notebook but state observations)
PPO_CHECKPOINT = ""          # set to an existing ppo.py checkpoint path to skip PPO training
PPO_SEED = 1
PPO_TIMESTEPS = 50_000_000
PPO_NUM_ENVS = 1024
PPO_NUM_STEPS = 16
PPO_MINIBATCHES = 32
PPO_EPOCHS = 8
PPO_EVAL_FREQ = 25           # iterations between ppo.py's own evaluations and checkpoints
PPO_CANDIDATES = 5           # evaluate this many of the latest checkpoints and keep the best
MIN_EXPERT_SUCCESS_ONCE = 0.80   # stop before BC if the expert is weaker than this on validation

# BC teacher
TEACHER_NAME = "bc_variable_goal_seed0"
SEED = 0
ITERS = 200_000
BATCH_SIZE = 1024
LR = 3e-4
EVAL_EVERY = 25_000
EVAL_EPISODES = 200          # 50 per quadrant on average
VAL_SEED = 1001
TEST_SEED = 5001

# Expert rollouts for BC
NUM_ENVS = 1024
EXPERT_EPISODES = 16_384                # 16 batches of 1,024
EXPERT_NOISE = [0.0, 0.1, 0.2, 0.3]     # action-noise std per batch, cycled; labels stay clean
EXPERT_SEED_BASE = 100_000
VAL_EPISODES = 1024                     # noise-free expert rollouts for validation action error
VAL_SEED_BASE = 300_000

SESSION = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + RUN_LABEL
subprocess.run(["nvidia-smi"], check=True)
print("Notebook Python:", sys.version)

## 1. Save outputs to Drive

Each session gets a new folder, so reruns never overwrite earlier results.

In [ ]:
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_ROOT = Path("/content/drive/MyDrive") / DRIVE_SUBDIR / SESSION
else:
    OUTPUT_ROOT = WORK_ROOT / "outputs" / SESSION
OUTPUT_ROOT.mkdir(parents=True, exist_ok=False)
LOG = OUTPUT_ROOT / "notebook.log"
print("Output directory:", OUTPUT_ROOT)


def run(command, show=None, **kwargs):
    """Run a command and append all output to notebook.log. Prints every line, or only lines
    matching the regex `show`. On failure, raises with the last 40 lines."""
    command = list(map(str, command))
    print("$", shlex.join(command), flush=True)
    pattern = re.compile(show) if show else None
    tail = []
    with LOG.open("a") as log:
        log.write("\n$ " + shlex.join(command) + "\n")
        with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, bufsize=1, **kwargs) as process:
            try:
                for line in process.stdout:
                    if pattern is None or pattern.search(line):
                        print(line, end="", flush=True)
                    log.write(line)
                    tail = (tail + [line.rstrip()])[-40:]
                code = process.wait()
            except BaseException:
                process.kill()
                raise
    if code:
        raise RuntimeError(f"Command failed (exit {code}). Log: {LOG}\n" + "\n".join(tail))

## 2. Install ManiSkill v3.0.1 under Python 3.11

Same pinned install as runs 1–2, plus TensorBoard, which ManiSkill's PPO script imports.

In [ ]:
run(["apt-get", "update", "-qq"])
run(["apt-get", "install", "-y", "-qq", "libvulkan1", "libvulkan-dev", "vulkan-tools", "libglvnd-dev"])
if not (MS_ROOT / ".git").exists():
    run(["git", "clone", "https://github.com/mani-skill/ManiSkill.git", MS_ROOT])
run(["git", "-C", MS_ROOT, "checkout", "--detach", MANISKILL_COMMIT])
assert subprocess.check_output(["git", "-C", str(MS_ROOT), "rev-parse", "HEAD"], text=True).strip() == MANISKILL_COMMIT

run([sys.executable, "-m", "pip", "install", "-q", "uv==0.9.11"])
TRAIN_ENV = WORK_ROOT / "python311"
TRAIN_PY = TRAIN_ENV / "bin/python"
if not TRAIN_PY.exists():
    run([sys.executable, "-m", "uv", "venv", "--python", "3.11", "--seed", TRAIN_ENV])
constraints = WORK_ROOT / "constraints.txt"
constraints.write_text("numpy==1.26.4\nopencv-python==4.11.0.86\nsapien==3.0.3\ntorch==2.6.0\n")
run([TRAIN_PY, "-m", "pip", "install", "-q", "torch==2.6.0", "--index-url", "https://download.pytorch.org/whl/cu124"])
run([TRAIN_PY, "-m", "pip", "install", "-q", "--constraint", constraints, "-e", MS_ROOT,
     "numpy", "sapien", "h5py", "tyro", "tensorboard", "imageio[ffmpeg]", "matplotlib"])

# Vulkan/EGL entries Colab is missing; needed for rendering evaluation videos.
for filename, library, vulkan in [
    ("/usr/share/vulkan/icd.d/nvidia_icd.json", "libGLX_nvidia.so.0", True),
    ("/usr/share/glvnd/egl_vendor.d/10_nvidia.json", "libEGL_nvidia.so.0", False),
]:
    path = Path(filename)
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        icd = {"library_path": library}
        if vulkan:
            icd["api_version"] = "1.2.155"
        path.write_text(json.dumps({"file_format_version": "1.0.0", "ICD": icd}))
os.environ["VK_ICD_FILENAMES"] = "/usr/share/vulkan/icd.d/nvidia_icd.json"
os.environ["MS_SKIP_ASSET_DOWNLOAD_PROMPT"] = "1"
os.environ["PYTHONUNBUFFERED"] = "1"
run([TRAIN_PY, "-c", "import torch, sapien, mani_skill, numpy; print('torch', torch.__version__, 'CUDA', torch.version.cuda); "
     "print('mani_skill', mani_skill.__version__); assert torch.cuda.is_available(); print(torch.cuda.get_device_name(0))"])
(OUTPUT_ROOT / "pip-freeze.txt").write_text(subprocess.check_output([str(TRAIN_PY), "-m", "pip", "freeze"], text=True))
PPO_SCRIPT = MS_ROOT / "examples/baselines/ppo/ppo.py"
assert PPO_SCRIPT.is_file()
print("Training Python:", TRAIN_PY)

## 3. Write the modules

`variable_goal_env.py` defines `PushT-VariableGoal-v1`. `pusht_state_bc.py` holds everything else
(`ppo`, `audit`, `check-env`, `collect`, `train`, `eval`). Repo copies live in `state_bc/`; this notebook is generated
from them by `state_bc/build_notebook.py`.

In [ ]:
%%writefile /content/pusht_state_bc/pusht_state_bc.py
"""Full-state behavioral cloning (BC) teachers for ManiSkill Push-T.

The default environment is PushT-VariableGoal-v1 (variable_goal_env.py): the goal T's position
changes every episode, and policies see it in the state. Use --env-id PushT-v1 for the original
fixed-goal task. --goal-quadrants limits which goal quadrants are sampled.

Subcommands (run with the ManiSkill training interpreter):

    ppo      Run ManiSkill's state PPO baseline (examples/baselines/ppo/ppo.py) with the
             variable-goal environment registered. Extra arguments go to ppo.py.
    audit    Inspect the state observation and action spaces; write observation_contract.json.
    check-env  Verify the success test follows each episode's goal (variable-goal env).
    prepare  Convert a replayed state-mode demo file (.h5 + .json) into train/validation
             transition files, split by whole episode.
    collect  Roll out a driver policy (the expert, or a BC checkpoint for DAgger) and label
             every visited state with the expert's deterministic action.
    train    Train one MLP BC teacher on one or more transition files, evaluating closed-loop
             success on validation seeds.
    eval     Evaluate a BC checkpoint, or a PPO checkpoint (--ppo), on a seed range, with
             success broken down by goal quadrant.

Transition files are .npz with obs [N, d_s], actions [N, d_a] and episode_ids [N],
plus a .json sidecar describing where they came from.

Pure data/model code is importable without ManiSkill, so it can be unit tested offline.
ManiSkill and gymnasium are imported lazily inside the simulator functions.
"""

from __future__ import annotations

import argparse
import json
import math
import time
from pathlib import Path

import numpy as np
import torch
from torch import nn

VARIABLE_GOAL_ENV_ID = "PushT-VariableGoal-v1"
ENV_ID = VARIABLE_GOAL_ENV_ID  # set by configure_env() from --env-id
ENV_KWARGS = {}                # set by configure_env() from --goal-quadrants / --start-quadrants
# Per-episode records exposed by PushT-VariableGoal-v1 and saved with collected data.
EPISODE_RECORDS = ("goal_quadrant", "goal_xy", "tip_start_quadrant", "tip_start_xy", "tee_start_quadrant",
                   "tip_ik_fallback")
OBS_MODE = "state"
CONTROL_MODE = "pd_joint_delta_pos"
SIM_BACKEND = "physx_cuda"
# Features whose training std is below this are only centered, not scaled. Near-constant
# fields (for example the fixed goal position) would otherwise turn simulator noise
# into very large normalized values.
MIN_SCALED_STD = 1e-3


# --------------------------------------------------------------------------------------
# Transition files
# --------------------------------------------------------------------------------------

def save_transitions(path, obs, actions, episode_ids, meta=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(path, obs=np.asarray(obs, np.float32), actions=np.asarray(actions, np.float32),
                        episode_ids=np.asarray(episode_ids, np.int64))
    meta = dict(meta or {}, transitions=int(len(obs)), episodes=int(len(np.unique(episode_ids))))
    path.with_suffix(".json").write_text(json.dumps(meta, indent=2))
    return meta


def load_transitions(paths):
    """Concatenate transition files; episode ids are made unique across files."""
    obs, actions, episode_ids, offset = [], [], [], 0
    for p in paths:
        d = np.load(p)
        obs.append(d["obs"])
        actions.append(d["actions"])
        ids = d["episode_ids"].astype(np.int64)
        episode_ids.append(ids - ids.min() + offset if len(ids) else ids)
        offset = (episode_ids[-1].max() + 1) if len(ids) else offset
    return np.concatenate(obs), np.concatenate(actions), np.concatenate(episode_ids)


# --------------------------------------------------------------------------------------
# Released demos
# --------------------------------------------------------------------------------------

def load_replayed_demos(h5_path: str | Path):
    """Return a list of episodes, each a dict with obs [T, d_s], actions [T, d_a], and metadata.

    The replayed file stores T+1 observations per episode; the last one has no action
    and is dropped, as in ManiSkill's own BC baseline.
    """
    import h5py

    h5_path = Path(h5_path)
    meta = json.loads(h5_path.with_suffix(".json").read_text())
    env_kwargs = meta["env_info"]["env_kwargs"]
    if env_kwargs.get("control_mode") not in (None, CONTROL_MODE):
        raise ValueError(f"Demo control mode {env_kwargs.get('control_mode')} != {CONTROL_MODE}")
    if env_kwargs.get("obs_mode") != OBS_MODE:
        raise ValueError(f"Demo obs mode {env_kwargs.get('obs_mode')} != {OBS_MODE}; replay with -o state")
    episodes = []
    with h5py.File(h5_path, "r") as f:
        for ep in meta["episodes"]:
            group = f[f"traj_{ep['episode_id']}"]
            obs = np.asarray(group["obs"], dtype=np.float32)
            actions = np.asarray(group["actions"], dtype=np.float32)
            if obs.ndim != 2 or actions.ndim != 2 or obs.shape[0] != actions.shape[0] + 1:
                raise ValueError(f"Unexpected shapes in traj_{ep['episode_id']}: obs {obs.shape}, actions {actions.shape}")
            success = np.asarray(group["success"]) if "success" in group else None
            episodes.append(dict(
                episode_id=int(ep["episode_id"]),
                episode_seed=ep.get("episode_seed"),
                obs=obs[:-1],
                # The simulator clips commands to the action space, so clipped labels are
                # exactly what the expert executed.
                actions=np.clip(actions, -1.0, 1.0),
                raw_actions=actions,
                success_at_end=bool(success[-1]) if success is not None else bool(ep.get("success", False)),
            ))
    return episodes, meta


def split_episodes(num_episodes: int, val_fraction: float, seed: int):
    """Split episode indices into train/validation sets, whole episodes only."""
    if not 0 < val_fraction < 1:
        raise ValueError("val_fraction must be in (0, 1)")
    rng = np.random.default_rng(seed)
    order = rng.permutation(num_episodes)
    num_val = max(1, int(round(num_episodes * val_fraction)))
    return np.sort(order[num_val:]), np.sort(order[:num_val])


def compute_state_stats(states: np.ndarray):
    mean = states.mean(axis=0)
    std = states.std(axis=0)
    scale = np.where(std < MIN_SCALED_STD, 1.0, std)
    return mean.astype(np.float32), std.astype(np.float32), scale.astype(np.float32)


def prepare(args):
    episodes, meta = load_replayed_demos(args.h5)
    if args.max_episodes:
        episodes = episodes[:args.max_episodes]
    train_idx, val_idx = split_episodes(len(episodes), args.val_fraction, args.split_seed)
    out = Path(args.out_dir)
    source = dict(source="released PPO demos, replayed to state obs", source_h5=str(args.h5),
                  source_env_info=meta["env_info"], source_commit=meta.get("commit_info"),
                  split_seed=args.split_seed, val_fraction=args.val_fraction)
    for name, idx in (("demos_train", train_idx), ("demos_val", val_idx)):
        eps = [episodes[i] for i in idx]
        save_transitions(out / f"{name}.npz",
                         np.concatenate([e["obs"] for e in eps]),
                         np.concatenate([e["actions"] for e in eps]),
                         np.concatenate([np.full(len(e["actions"]), e["episode_id"]) for e in eps]),
                         dict(source, split=name, episode_ids=[e["episode_id"] for e in eps]))
    raw = np.concatenate([e["raw_actions"] for e in episodes])
    summary = dict(
        state_dim=int(episodes[0]["obs"].shape[1]), action_dim=int(raw.shape[1]),
        episodes_in_file=len(meta["episodes"]), episodes_replayed=len(episodes),
        train_episodes=len(train_idx), val_episodes=len(val_idx),
        raw_action_min=raw.min(0).round(4).tolist(), raw_action_max=raw.max(0).round(4).tolist(),
        fraction_raw_actions_outside_bounds=float((np.abs(raw) > 1).mean()),
        replayed_success_at_end=float(np.mean([e["success_at_end"] for e in episodes])),
    )
    (out / "demos_summary.json").write_text(json.dumps(summary, indent=2))
    print(json.dumps(summary, indent=2))


# --------------------------------------------------------------------------------------
# Models
# --------------------------------------------------------------------------------------

class BCPolicy(nn.Module):
    """MLP teacher: normalized state -> action in [-1, 1].

    Normalization statistics are registered buffers, so a checkpoint carries its own
    preprocessing and the policy accepts raw environment observations.
    """

    def __init__(self, state_dim, action_dim, state_mean, state_scale, hidden=256, depth=2, tanh_output=True):
        super().__init__()
        layers, width = [], state_dim
        for _ in range(depth):
            layers += [nn.Linear(width, hidden), nn.ReLU()]
            width = hidden
        layers.append(nn.Linear(width, action_dim))
        self.net = nn.Sequential(*layers)
        self.tanh_output = tanh_output
        self.register_buffer("state_mean", torch.as_tensor(state_mean, dtype=torch.float32))
        self.register_buffer("state_scale", torch.as_tensor(state_scale, dtype=torch.float32))
        self.config = dict(state_dim=state_dim, action_dim=action_dim, hidden=hidden, depth=depth, tanh_output=tanh_output)

    def forward(self, raw_state):
        out = self.net((raw_state - self.state_mean) / self.state_scale)
        return torch.tanh(out) if self.tanh_output else out

    @torch.no_grad()
    def act(self, raw_state):
        return self(raw_state).clamp(-1.0, 1.0)


def save_policy(policy: BCPolicy, path, extra=None):
    torch.save({"config": policy.config, "state_dict": policy.state_dict(), "extra": extra or {}}, path)


def load_policy(path, device="cpu") -> BCPolicy:
    ckpt = torch.load(path, map_location=device, weights_only=False)
    cfg = ckpt["config"]
    policy = BCPolicy(cfg["state_dim"], cfg["action_dim"], np.zeros(cfg["state_dim"]), np.ones(cfg["state_dim"]),
                      hidden=cfg["hidden"], depth=cfg["depth"], tanh_output=cfg["tanh_output"])
    policy.load_state_dict(ckpt["state_dict"])
    return policy.to(device).eval()


class ReleasedPPOActor(nn.Module):
    """Deterministic mean of ManiSkill's state PPO actor (examples/baselines/ppo/ppo.py)."""

    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.actor_mean = nn.Sequential(
            nn.Linear(state_dim, 256), nn.Tanh(), nn.Linear(256, 256), nn.Tanh(),
            nn.Linear(256, 256), nn.Tanh(), nn.Linear(256, action_dim))

    @torch.no_grad()
    def act(self, raw_state):
        return self.actor_mean(raw_state).clamp(-1.0, 1.0)


def load_released_ppo(path, state_dim, action_dim, device="cpu"):
    weights = torch.load(path, map_location=device, weights_only=False)
    actor = ReleasedPPOActor(state_dim, action_dim)
    actor.load_state_dict({k: v for k, v in weights.items() if k.startswith("actor_mean.")})
    return actor.to(device).eval()


# --------------------------------------------------------------------------------------
# Rollouts (shared by evaluation and data collection)
# --------------------------------------------------------------------------------------

@torch.no_grad()
def rollout(env, max_steps, driver, seed, labeler=None, noise_std=0.0, generator=None):
    """Run one batch of episodes, one per parallel env, for max_steps with no resets.

    driver picks the executed action; Gaussian noise (std on the [-1, 1] action scale) is
    added after labeling, so labels stay the labeler's clean actions. If labeler is given,
    returns obs [n, T, d_s] and labels [n, T, d_a] on the CPU, episode-major.
    """
    obs, _ = env.reset(seed=seed)
    n = obs.shape[0]
    device = obs.device
    base = getattr(env, "base_env", None)
    # Per-episode start/goal records from PushT-VariableGoal-v1, read right after reset.
    records = {name: getattr(base, name).clone() for name in EPISODE_RECORDS if hasattr(base, name)}
    quadrant = records.get("goal_quadrant")
    success_once = torch.zeros(n, dtype=torch.bool, device=device)
    first_success = torch.full((n,), -1, dtype=torch.long, device=device)
    obs_buf, label_buf = [], []
    success = success_once
    for t in range(max_steps):
        o = obs.float()
        action = driver.act(o)
        if labeler is not None:
            label = action if labeler is driver else labeler.act(o)
            obs_buf.append(o.cpu())
            label_buf.append(label.cpu())
        if noise_std > 0:
            noise = torch.randn(action.shape, device=action.device, generator=generator)
            action = (action + noise_std * noise).clamp(-1.0, 1.0)
        obs, _, _, _, info = env.step(action)
        success = info["success"].bool()
        newly = (first_success < 0) & success
        first_success = torch.where(newly, torch.full_like(first_success, t + 1), first_success)
        success_once |= success
    steps = first_success[first_success > 0].float()
    result = {
        "episodes": int(n), "seed": int(seed), "noise_std": float(noise_std),
        "success_once": success_once.float().mean().item(),
        "success_at_end": success.float().mean().item(),
        "mean_steps_to_first_success": steps.mean().item() if len(steps) else None,
    }
    if quadrant is not None:
        result["by_goal_quadrant"] = {
            f"Q{q}": {"episodes": int(m.sum()),
                      "success_once": success_once[m].float().mean().item(),
                      "success_at_end": success[m].float().mean().item()}
            for q in range(1, 5) if (m := quadrant == q).any()}
    if "tip_ik_fallback" in records:
        result["tip_ik_fallback_rate"] = records["tip_ik_fallback"].float().mean().item()
    if labeler is not None:
        for name, value in records.items():
            result[f"{name}_per_episode"] = value.cpu().tolist()
    if labeler is None:
        return result
    return result, torch.stack(obs_buf, 1).numpy(), torch.stack(label_buf, 1).numpy()


# --------------------------------------------------------------------------------------
# Training
# --------------------------------------------------------------------------------------

@torch.no_grad()
def action_mse(policy, obs, actions, device, batch=65536):
    total, count = 0.0, 0
    for i in range(0, len(obs), batch):
        o = torch.as_tensor(obs[i:i + batch], device=device)
        a = torch.as_tensor(actions[i:i + batch], device=device)
        total += torch.nn.functional.mse_loss(policy(o), a, reduction="sum").item()
        count += a.numel()
    return total / max(count, 1)


def train(args):
    torch.manual_seed(args.seed)
    np.random.seed(args.seed)
    device = torch.device(args.device)
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)

    obs_np, act_np, ep_np = load_transitions(args.train_data)
    val_obs, val_act, _ = load_transitions([args.val_data])
    mean, std, scale = compute_state_stats(obs_np)
    data_info = dict(train_files=[str(p) for p in args.train_data], val_file=str(args.val_data),
                     train_transitions=int(len(obs_np)), train_episodes=int(len(np.unique(ep_np))),
                     val_transitions=int(len(val_obs)),
                     features_only_centered=np.flatnonzero(std < MIN_SCALED_STD).tolist())
    (out / "data_info.json").write_text(json.dumps(data_info, indent=2))
    print(f"Training on {data_info['train_transitions']:,} transitions from "
          f"{data_info['train_episodes']:,} episodes in {len(args.train_data)} file(s)", flush=True)

    train_obs = torch.as_tensor(obs_np, device=device)
    train_act = torch.as_tensor(act_np, device=device).clamp(-1.0, 1.0)
    if args.tanh_output:
        # tanh cannot reach exactly +-1; keep targets just inside the bound.
        train_act = train_act.clamp(-0.999, 0.999)
    state_dim, action_dim = train_obs.shape[1], train_act.shape[1]
    policy = BCPolicy(state_dim, action_dim, mean, scale,
                      hidden=args.hidden, depth=args.depth, tanh_output=args.tanh_output).to(device)
    opt = torch.optim.Adam(policy.parameters(), lr=args.lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=args.iters) if args.cosine else None
    gen = torch.Generator(device=device).manual_seed(args.seed)

    evaluator = Evaluator(num_envs=args.eval_episodes, device=device) if args.eval_every > 0 else None

    fields = ("iter", "train_mse", "val_mse", "lr", "success_once", "success_at_end", "seconds")
    log = (out / "train_log.csv").open("w")
    log.write(",".join(fields) + "\n")
    best = (-1.0, -1.0)
    start = time.time()
    running, samples = 0.0, 0
    for it in range(1, args.iters + 1):
        idx = torch.randint(0, len(train_obs), (args.batch_size,), device=device, generator=gen)
        loss = torch.nn.functional.mse_loss(policy(train_obs[idx]), train_act[idx])
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        if sched:
            sched.step()
        if it % 100 == 0:
            running += loss.item()
            samples += 1

        last = it == args.iters
        if it % args.log_every == 0 or last:
            policy.eval()
            val_mse = action_mse(policy, val_obs, val_act.clip(-1, 1), device)
            row = dict(iter=it, train_mse=running / max(samples, 1), val_mse=val_mse,
                       lr=opt.param_groups[0]["lr"], success_once="", success_at_end="")
            running, samples = 0.0, 0
            if evaluator and (it % args.eval_every == 0 or last):
                res = evaluator.run(policy, seed=args.val_seed)
                row.update(success_once=res["success_once"], success_at_end=res["success_at_end"])
                save_policy(policy, out / f"ckpt_{it}.pt", {"iter": it, **res})
                key = (res["success_at_end"], res["success_once"])
                if key > best:
                    best = key
                    save_policy(policy, out / "best.pt", {"iter": it, **res})
                    (out / "best.json").write_text(json.dumps({"iter": it, **res}, indent=2))
            policy.train()
            row["seconds"] = round(time.time() - start, 1)
            log.write(",".join(str(row[k]) for k in fields) + "\n")
            log.flush()
            msg = f"iter {it:>7}  train_mse {row['train_mse']:.5f}  val_mse {val_mse:.5f}"
            if row["success_once"] != "":
                msg += f"  val success_once {row['success_once']:.0%}  success_at_end {row['success_at_end']:.0%}"
            print(msg, flush=True)
    log.close()
    save_policy(policy, out / "final.pt", {"iter": args.iters})
    if evaluator is None:
        save_policy(policy, out / "best.pt", {"iter": args.iters, "note": "no closed-loop eval; best = final"})
    else:
        evaluator.close()
    print(f"Done in {(time.time() - start) / 60:.1f} min. Best validation (end, once): {best}")


# --------------------------------------------------------------------------------------
# Simulator: audit, evaluation, collection
# --------------------------------------------------------------------------------------

def configure_env(env_id, goal_quadrants=None, start_quadrants=None):
    """Select the environment for this process (registration happens in make_env)."""
    global ENV_ID, ENV_KWARGS
    ENV_ID, ENV_KWARGS = env_id, {}
    if env_id == VARIABLE_GOAL_ENV_ID:
        if goal_quadrants:
            ENV_KWARGS["goal_quadrants"] = tuple(goal_quadrants)
        if start_quadrants:
            ENV_KWARGS["start_quadrants"] = tuple(start_quadrants)
    elif goal_quadrants or start_quadrants:
        raise ValueError("--goal-quadrants and --start-quadrants only apply to " + VARIABLE_GOAL_ENV_ID)


def make_env(num_envs, obs_mode=OBS_MODE, **kwargs):
    import gymnasium as gym
    import mani_skill.envs  # noqa: F401  (registers environments)

    if ENV_ID == VARIABLE_GOAL_ENV_ID:
        import variable_goal_env  # registers PushT-VariableGoal-v1
        if variable_goal_env.PushTVariableGoalEnv is None:
            raise ImportError("variable_goal_env could not import ManiSkill; the environment is not registered")
    return gym.make(ENV_ID, num_envs=num_envs, obs_mode=obs_mode, control_mode=CONTROL_MODE,
                    sim_backend=SIM_BACKEND, render_mode="rgb_array", **ENV_KWARGS, **kwargs)


def run_ppo(argv):
    """Run ManiSkill's ppo.py in this process after registering the variable-goal environment.
    ppo.py writes runs/<exp-name>/ under the current directory."""
    import runpy
    import sys

    p = argparse.ArgumentParser(prog="pusht_state_bc.py ppo", allow_abbrev=False)
    p.add_argument("--script", required=True, help="path to ManiSkill examples/baselines/ppo/ppo.py")
    known, rest = p.parse_known_args(argv)
    import variable_goal_env  # noqa: F401  (registers PushT-VariableGoal-v1 for ppo.py's gym.make)
    sys.argv = [known.script] + rest
    runpy.run_path(known.script, run_name="__main__")


def episode_length(env):
    """Max episode steps. Vectorized GPU envs leave env.spec.max_episode_steps as None,
    so use ManiSkill's own lookup (as its PPO baseline does), with PushT-v1's registered 100 as fallback."""
    from mani_skill.utils import gym_utils

    steps = gym_utils.find_max_episode_steps_value(env)
    return int(steps) if steps is not None else 100


def make_vector_env(num_envs, video_dir=None, **kwargs):
    """Fixed-length episodes: no auto-reset (it would replace the final step's info) and
    terminations ignored, matching the evaluation convention of ManiSkill's PPO baseline."""
    from mani_skill.vector.wrappers.gymnasium import ManiSkillVectorEnv

    env = make_env(num_envs, **kwargs)
    max_steps = episode_length(env)
    if video_dir is not None:
        from mani_skill.utils.wrappers.record import RecordEpisode
        env = RecordEpisode(env, output_dir=str(video_dir), save_trajectory=False,
                            max_steps_per_video=max_steps, video_fps=30)
    return ManiSkillVectorEnv(env, num_envs, auto_reset=False, ignore_terminations=True, record_metrics=True), max_steps


def flatten_with_names(tree, prefix=""):
    """Flatten a nested state dict in insertion order, as ManiSkill does; return names and widths."""
    fields = []
    for key, value in tree.items():
        name = f"{prefix}{key}"
        if isinstance(value, dict):
            fields += flatten_with_names(value, name + "/")
        else:
            width = 1 if value.ndim == 1 else int(value.shape[-1])
            fields.append((name, width))
    return fields


def audit(args):
    import mani_skill

    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    env = make_env(1)
    try:
        obs, info = env.reset(seed=0)
        tree = env.unwrapped.get_obs(info, unflattened=True)
        fields = flatten_with_names(tree)
        width = sum(w for _, w in fields)
        assert obs.shape == (1, width), f"Flat obs {tuple(obs.shape)} does not match fields totalling {width}"
        offsets, start = {}, 0
        for name, w in fields:
            offsets[name] = [start, start + w]
            start += w
        if not any("obj_pose" in n for n, _ in fields):
            raise AssertionError("State observation lacks obj_pose; this is not the full-state setting")
        space = env.unwrapped.single_action_space
        contract = {
            "env_id": ENV_ID, "env_kwargs": {k: list(v) for k, v in ENV_KWARGS.items()}, "obs_mode": OBS_MODE, "control_mode": CONTROL_MODE, "sim_backend": SIM_BACKEND,
            "mani_skill_version": getattr(mani_skill, "__version__", "unknown"),
            "state_dim": width, "state_fields_in_order": offsets,
            "action_dim": int(np.prod(space.shape)),
            "action_low": space.low.tolist(), "action_high": space.high.tolist(),
            "control_freq": getattr(env.unwrapped, "control_freq", None),
            "sim_freq": getattr(env.unwrapped, "sim_freq", None),
            "max_episode_steps": episode_length(env),
            "robot_uids": str(getattr(env.unwrapped, "robot_uids", None)),
            "success_rule": "evaluate(): T overlap with goal >= intersection_thresh",
            "intersection_thresh": getattr(env.unwrapped, "intersection_thresh", None),
        }
        if not (np.allclose(space.low, -1) and np.allclose(space.high, 1)):
            print("WARNING: action space is not [-1, 1]; the BC tanh head assumes normalized actions.")
        (out / "observation_contract.json").write_text(json.dumps(contract, indent=2))
        print(json.dumps(contract, indent=2))
    finally:
        env.close()


def check_env(args):
    """Check the success test against each episode's own goal, without stepping physics.

    Puts a stand-in T exactly on every episode's goal pose and runs the environment's overlap
    computation: all must count as success. Then shifts it 5 cm: none may. With PushT-v1's single
    fixed goal transform, varied goals would fail the first check. Also reports goal ranges and quadrants.
    """
    import types

    env = make_env(args.num_envs)
    try:
        env.reset(seed=args.seed)
        base = env.unwrapped
        goal_p, goal_q = base.goal_tee.pose.p.clone(), base.goal_tee.pose.q.clone()
        real_tee = base.tee

        def overlap(p):
            base.tee = types.SimpleNamespace(pose=types.SimpleNamespace(p=p, q=goal_q))
            try:
                return base.pseudo_render_intersection()
            finally:
                base.tee = real_tee

        at_goal = overlap(goal_p)
        shifted = goal_p.clone()
        shifted[:, 0] += 0.05
        off_goal = overlap(shifted)
        thresh = base.intersection_thresh
        quadrant = getattr(base, "goal_quadrant", None)
        report = {
            "env_id": ENV_ID, "num_envs": args.num_envs,
            "goal_x_range": [goal_p[:, 0].min().item(), goal_p[:, 0].max().item()],
            "goal_y_range": [goal_p[:, 1].min().item(), goal_p[:, 1].max().item()],
            "goal_quadrant_counts": ({f"Q{q}": int((quadrant == q).sum()) for q in range(1, 5)}
                                     if quadrant is not None else None),
            "overlap_with_T_on_goal_min": at_goal.min().item(),
            "overlap_with_T_shifted_5cm_max": off_goal.max().item(),
            "intersection_thresh": thresh,
        }
        passed = bool((at_goal >= thresh).all() and (off_goal < thresh).all())
        if hasattr(base, "tip_start_xy") and getattr(base, "random_tip_start", False):
            # Where the stick tip actually is after reset, versus where IK placed it.
            tcp_xy = base.agent.tcp.pose.p[:, :2]
            placed = ~base.tip_ik_fallback
            gap = torch.linalg.norm(tcp_xy - base.tip_start_xy, dim=1)
            tee_dist = torch.linalg.norm(tcp_xy - base.tee.pose.p[:, :2], dim=1)
            tip_q = base.tip_start_quadrant
            report.update({
                "tip_x_range": [tcp_xy[:, 0].min().item(), tcp_xy[:, 0].max().item()],
                "tip_y_range": [tcp_xy[:, 1].min().item(), tcp_xy[:, 1].max().item()],
                "tip_start_quadrant_counts": {f"Q{q}": int((tip_q == q).sum()) for q in range(1, 5)},
                "tee_start_quadrant_counts": {f"Q{q}": int((base.tee_start_quadrant == q).sum()) for q in range(1, 5)},
                "tip_ik_fallback_rate": base.tip_ik_fallback.float().mean().item(),
                "tip_grid_reachable_fraction": base._ik["valid_fraction"] if getattr(base, "_ik", None) else None,
                "tip_ik_error_max_m": base.tip_ik_error.max().item(),
                "tip_vs_ik_gap_max_m": gap[placed].max().item() if placed.any() else None,
                "tip_height_range_m": [base.agent.tcp.pose.p[:, 2].min().item(), base.agent.tcp.pose.p[:, 2].max().item()],
                "tip_to_tee_min_m": tee_dist.min().item(),
                "goal_vs_tip_quadrant_agreement": (tip_q == quadrant).float().mean().item() if quadrant is not None else None,
            })
            # Placement must match the simulator's own kinematics (catches a wrong base offset or chain)
            # and fall back rarely.
            passed = passed and report["tip_ik_fallback_rate"] <= 0.05 and (
                report["tip_vs_ik_gap_max_m"] is None or report["tip_vs_ik_gap_max_m"] < 0.02)
        report["passed"] = passed
    finally:
        env.close()
    print(json.dumps(report, indent=2))
    if args.out:
        Path(args.out).write_text(json.dumps(report, indent=2))
    if not report["passed"]:
        raise SystemExit("Environment check FAILED: success is not measured against each episode's goal.")


class Evaluator:
    """Closed-loop evaluation on a fixed batch of seeded initial states."""

    def __init__(self, num_envs, device, video_dir=None):
        self.env, self.max_steps = make_vector_env(num_envs, video_dir=video_dir, reconfiguration_freq=1)
        self.num_envs = num_envs
        self.device = device

    def run(self, policy, seed):
        return rollout(self.env, self.max_steps, policy, seed)

    def close(self):
        self.env.close()


def load_expert(path, env, device):
    state_dim = int(np.prod(env.single_observation_space.shape))
    action_dim = int(np.prod(env.single_action_space.shape))
    return load_released_ppo(path, state_dim, action_dim, device)


def evaluate(args):
    device = torch.device(args.device)
    video_dir = Path(args.video_dir) if args.video_dir else None
    evaluator = Evaluator(args.episodes, device, video_dir=video_dir)
    try:
        if args.ppo:
            policy = load_expert(args.checkpoint, evaluator.env, device)
        else:
            policy = load_policy(args.checkpoint, device)
        res = evaluator.run(policy, seed=args.seed)
    finally:
        evaluator.close()
    res["checkpoint"] = str(args.checkpoint)
    print(json.dumps(res, indent=2))
    if args.out:
        Path(args.out).write_text(json.dumps(res, indent=2))
    return res


def collect(args):
    """Roll out a driver and label every visited state with the expert.

    driver=expert (optionally with action noise) gives more, and slightly off-course,
    expert demonstrations. driver=bc gives one round of DAgger data: states the BC policy
    actually reaches, labeled with what the expert would do there.
    """
    device = torch.device(args.device)
    env, max_steps = make_vector_env(args.num_envs)
    try:
        expert = load_expert(args.expert, env, device)
        driver = expert if args.driver == "expert" else load_policy(args.driver_checkpoint, device)
        noise_levels = args.noise_std or [0.0]
        gen = torch.Generator(device=device).manual_seed(args.seed_base)
        batches = math.ceil(args.episodes / args.num_envs)
        all_obs, all_labels, per_batch, all_noise = [], [], [], []
        per_episode = {}
        for b in range(batches):
            sigma = noise_levels[b % len(noise_levels)]
            res, obs, labels = rollout(env, max_steps, driver, seed=args.seed_base + b,
                                       labeler=expert, noise_std=sigma, generator=gen)
            all_obs.append(obs)
            all_labels.append(labels)
            for key in [k for k in res if k.endswith("_per_episode")]:
                per_episode.setdefault(key, []).extend(res.pop(key))
            all_noise += [sigma] * obs.shape[0]
            per_batch.append(res)
            print(f"batch {b + 1}/{batches}  noise {sigma:.2f}  driver success_once {res['success_once']:.0%}  "
                  f"success_at_end {res['success_at_end']:.0%}", flush=True)
    finally:
        env.close()
    obs = np.concatenate(all_obs)[:args.episodes]       # [episodes, T, d_s]
    labels = np.concatenate(all_labels)[:args.episodes]
    n, T = obs.shape[:2]
    meta = dict(
        driver=args.driver, driver_checkpoint=args.driver_checkpoint or None, expert=str(args.expert),
        labels="expert deterministic action, clipped to [-1, 1]", noise_std_cycle=noise_levels,
        env_id=ENV_ID, env_kwargs={k: list(v) for k, v in ENV_KWARGS.items()},
        seed_base=args.seed_base, num_envs=args.num_envs, steps_per_episode=int(T), batches=per_batch,
        driver_success_once=float(np.mean([r["success_once"] for r in per_batch])),
        driver_success_at_end=float(np.mean([r["success_at_end"] for r in per_batch])),
        noise_std_per_episode=all_noise[:n],
    )
    for key, values in per_episode.items():
        meta[key] = values[:n]
    meta = save_transitions(args.out, obs.reshape(n * T, -1), labels.reshape(n * T, -1),
                            np.repeat(np.arange(n), T), meta)
    print(json.dumps({k: v for k, v in meta.items() if not k.endswith("per_episode") and k != "batches"}, indent=2))


# --------------------------------------------------------------------------------------
# CLI
# --------------------------------------------------------------------------------------

def main(argv=None):
    import sys

    argv = sys.argv[1:] if argv is None else list(argv)
    if argv and argv[0] == "ppo":
        return run_ppo(argv[1:])

    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = p.add_subparsers(dest="cmd", required=True)
    sub.add_parser("ppo", help="run ManiSkill ppo.py on the variable-goal env (see run_ppo)")
    default_device = "cuda" if torch.cuda.is_available() else "cpu"
    env_opts = argparse.ArgumentParser(add_help=False)
    env_opts.add_argument("--env-id", default=VARIABLE_GOAL_ENV_ID)
    env_opts.add_argument("--goal-quadrants", type=int, nargs="+", default=None,
                          help="goal quadrants to sample (variable-goal env only); default all four")
    env_opts.add_argument("--start-quadrants", type=int, nargs="+", default=None,
                          help="stick-tip start quadrants to sample (variable-goal env only); default all four")

    a = sub.add_parser("audit", parents=[env_opts])
    a.add_argument("--out", required=True)

    ce = sub.add_parser("check-env", parents=[env_opts])
    ce.add_argument("--num-envs", type=int, default=256)
    ce.add_argument("--seed", type=int, default=0)
    ce.add_argument("--out", default="")

    pr = sub.add_parser("prepare")
    pr.add_argument("--h5", required=True)
    pr.add_argument("--out-dir", required=True)
    pr.add_argument("--val-fraction", type=float, default=0.1)
    pr.add_argument("--split-seed", type=int, default=0)
    pr.add_argument("--max-episodes", type=int, default=0, help="0 = use all episodes")

    c = sub.add_parser("collect", parents=[env_opts])
    c.add_argument("--expert", required=True, help="PPO checkpoint (ManiSkill ppo.py format) used for labels")
    c.add_argument("--driver", choices=("expert", "bc"), default="expert")
    c.add_argument("--driver-checkpoint", default="", help="BC checkpoint when --driver bc")
    c.add_argument("--episodes", type=int, required=True)
    c.add_argument("--num-envs", type=int, default=1024)
    c.add_argument("--noise-std", type=float, nargs="*", default=[0.0],
                   help="action noise std per batch, cycled; labels stay noise-free")
    c.add_argument("--seed-base", type=int, required=True)
    c.add_argument("--out", required=True)
    c.add_argument("--device", default=default_device)

    t = sub.add_parser("train", parents=[env_opts])
    t.add_argument("--train-data", nargs="+", required=True)
    t.add_argument("--val-data", required=True)
    t.add_argument("--out", required=True)
    t.add_argument("--seed", type=int, default=0)
    t.add_argument("--iters", type=int, default=100_000)
    t.add_argument("--batch-size", type=int, default=1024)
    t.add_argument("--lr", type=float, default=3e-4)
    t.add_argument("--cosine", action=argparse.BooleanOptionalAction, default=True)
    t.add_argument("--hidden", type=int, default=256)
    t.add_argument("--depth", type=int, default=2)
    t.add_argument("--tanh-output", action=argparse.BooleanOptionalAction, default=True)
    t.add_argument("--log-every", type=int, default=1000)
    t.add_argument("--eval-every", type=int, default=10_000, help="0 disables closed-loop eval")
    t.add_argument("--eval-episodes", type=int, default=100)
    t.add_argument("--val-seed", type=int, default=1001)
    t.add_argument("--device", default=default_device)

    e = sub.add_parser("eval", parents=[env_opts])
    e.add_argument("--checkpoint", required=True)
    e.add_argument("--ppo", "--released-ppo", dest="ppo", action="store_true",
                   help="checkpoint is a ManiSkill ppo.py checkpoint (released or our own)")
    e.add_argument("--episodes", type=int, default=100)
    e.add_argument("--seed", type=int, default=2001)
    e.add_argument("--out", default="")
    e.add_argument("--video-dir", default="")
    e.add_argument("--device", default=default_device)

    args = p.parse_args(argv)
    if args.cmd == "collect" and args.driver == "bc" and not args.driver_checkpoint:
        p.error("--driver bc needs --driver-checkpoint")
    if hasattr(args, "env_id"):
        configure_env(args.env_id, args.goal_quadrants, args.start_quadrants)
    {"audit": audit, "check-env": check_env, "prepare": prepare, "collect": collect, "train": train,
     "eval": evaluate}[args.cmd](args)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/pusht_state_bc/variable_goal_env.py
"""PushT-VariableGoal-v1: ManiSkill PushT-v1 with a random goal and random stick-tip start per episode.

Changes from PushT-v1 (ManiSkill v3.0.1):
- Goal: the goal T's position is sampled every episode within +-GOAL_HALF_WIDTH (x and y) of
  PushT-v1's fixed goal (the "centre"). Its rotation stays fixed. Quadrants around the centre, world
  frame: Q1 (+x, +y), Q2 (-x, +y), Q3 (-x, -y), Q4 (+x, -y). `goal_quadrants` limits which are
  sampled (all four for teachers; Q1-Q3 for transformer training).
- Success check: uses a per-environment goal transform. PushT-v1 computes one fixed transform at load
  time, which would score every episode against the original goal.
- Stick-tip start: sampled within +-TIP_HALF_WIDTH of the centre, in a quadrant allowed by
  `start_quadrants` (default all four), at least TIP_CLEARANCE from the movable T's centre. Joint
  angles come from a lookup table: inverse kinematics (damped least squares, on the GPU) is solved once
  per process for a TIP_GRID x TIP_GRID grid of cell centres covering the square, at PushT-v1's rest tip
  height and stick orientation. Each reset uses the cell containing the sampled point, so the tip starts
  within half a cell diagonal (about 3 mm) of it, and resets cost no IK. Cells the solver cannot reach
  within IK_TOLERANCE are avoided; if an episode has no usable cell, it keeps PushT-v1's start.
  `random_tip_start=False` restores PushT-v1's fixed tip start.
- Movable T: PushT-v1's original absolute spawn region and random rotation (unchanged).
- Starts are sampled independently of the goal, so a start never reveals the goal.
- Observations are unchanged: state mode still includes `extra/goal_pos`, which now varies.

Per-episode records on the env: goal_xy, goal_quadrant, tip_start_xy, tip_start_quadrant,
tee_start_quadrant, tip_ik_error, tip_ik_fallback. The lookup table's reachable fraction is in
env._ik["valid_fraction"] after the first reset.

Importing this module registers the environment.
"""

import numpy as np
import torch

ENV_ID = "PushT-VariableGoal-v1"
GOAL_HALF_WIDTH = 0.10   # metres; goals within +-10 cm of the centre in x and y
TIP_HALF_WIDTH = 0.15    # metres; stick-tip starts within +-15 cm of the centre in x and y
TIP_CLEARANCE = 0.15     # metres from the T's centre of mass; the T reaches about 0.14 m from it
TIP_CANDIDATES = 32      # rejection-sampling candidates per episode
TIP_GRID = 64            # lookup table resolution per axis (cells of about 4.7 mm)
IK_ITERATIONS = 200      # one-time solve for the lookup table
IK_TOLERANCE = 0.01      # metres; larger position error falls back to PushT-v1's start
ROBOT_BASE_XYZ = (-0.615, 0.0, 0.0)  # PushT-v1 places the robot base here, unrotated
QUADRANT_SIGNS = {1: (1.0, 1.0), 2: (-1.0, 1.0), 3: (-1.0, -1.0), 4: (1.0, -1.0)}


def quadrant_of(offset_xy):
    """Quadrant (1-4) of offsets [n, 2] from the centre. Zero counts as positive."""
    pos_x, pos_y = offset_xy[:, 0] >= 0, offset_xy[:, 1] >= 0
    q = torch.full((offset_xy.shape[0],), 3, dtype=torch.long, device=offset_xy.device)
    q[pos_x & pos_y] = 1
    q[~pos_x & pos_y] = 2
    q[pos_x & ~pos_y] = 4
    return q


def sample_goal_offsets(n, quadrants, half_width=GOAL_HALF_WIDTH, generator=None, device=None):
    """Uniform over the allowed quadrants, then uniform within the chosen quadrant's square."""
    allowed = torch.as_tensor(sorted(quadrants), device=device)
    pick = allowed[torch.randint(len(allowed), (n,), generator=generator, device=device)]
    signs = torch.as_tensor([QUADRANT_SIGNS[int(q)] for q in range(1, 5)], device=device)[pick - 1]
    mags = torch.rand((n, 2), generator=generator, device=device) * half_width
    return signs * mags, pick


def tip_grid_offsets(half_width=TIP_HALF_WIDTH, n=TIP_GRID, device=None):
    """Cell centres [n*n, 2] of an n x n grid over [-half_width, half_width]^2; index = iy * n + ix.
    No centre lies on an axis, so a cell's quadrant equals the quadrant of any point inside it."""
    centres = (torch.arange(n, device=device, dtype=torch.float32) + 0.5) * (2 * half_width / n) - half_width
    gy, gx = torch.meshgrid(centres, centres, indexing="ij")
    return torch.stack([gx.reshape(-1), gy.reshape(-1)], dim=1)


def grid_index(offsets_xy, half_width=TIP_HALF_WIDTH, n=TIP_GRID):
    """Index of the grid cell containing each offset [m, 2]."""
    cell = 2 * half_width / n
    ix = ((offsets_xy[:, 0] + half_width) / cell).floor().long().clamp(0, n - 1)
    iy = ((offsets_xy[:, 1] + half_width) / cell).floor().long().clamp(0, n - 1)
    return iy * n + ix


def choose_tip_cells(tee_offset_xy, quadrants, valid, grid_offsets, half_width=TIP_HALF_WIDTH, n=TIP_GRID,
                     clearance=TIP_CLEARANCE, candidates=TIP_CANDIDATES, generator=None):
    """Pick a start cell per episode: sample points uniformly in an allowed quadrant, map each to its
    cell, keep the first cell that is reachable (valid) and at least `clearance` from the T. If none
    qualifies, keep the reachable candidate farthest from the T, else the farthest candidate.
    Returns cell indices and their distance to the T."""
    m, device = tee_offset_xy.shape[0], tee_offset_xy.device
    offs, _ = sample_goal_offsets(m * candidates, quadrants, half_width, generator, device)
    idx = grid_index(offs, half_width, n).view(m, candidates)
    dist = torch.linalg.norm(grid_offsets[idx] - tee_offset_xy.unsqueeze(1), dim=-1)
    reachable = valid[idx]
    ok = reachable & (dist >= clearance)
    fallback = torch.where(reachable.any(1), torch.where(reachable, dist, torch.full_like(dist, -1.0)).argmax(1),
                           dist.argmax(1))
    choice = torch.where(ok.any(1), ok.float().argmax(1), fallback)
    rows = torch.arange(m, device=device)
    return idx[rows, choice], dist[rows, choice]


def orientation_error(r_current, r_target):
    """Small-angle rotation error [n, 3] taking r_current to r_target (both [n, 3, 3])."""
    return 0.5 * torch.cross(r_current, r_target, dim=1).sum(-1)


def dls_ik(fk, jacobian, q0, target_pos, target_rot, lower, upper, iterations=IK_ITERATIONS, damping=0.05,
           max_step=0.2):
    """Batched damped-least-squares IK for position and orientation.

    fk(q) -> (pos [n, 3], rot [n, 3, 3]); jacobian(q) -> [n, 6, dof] (linear rows first).
    Returns joint positions and the final position error in metres.
    """
    q = q0.clone()
    eye = torch.eye(6, device=q.device, dtype=q.dtype)
    for _ in range(iterations):
        pos, rot = fk(q)
        err = torch.cat([target_pos - pos, orientation_error(rot, target_rot)], dim=1)
        J = jacobian(q)
        JJt = J @ J.transpose(1, 2) + (damping ** 2) * eye
        dq = (J.transpose(1, 2) @ torch.linalg.solve(JJt, err.unsqueeze(-1))).squeeze(-1)
        dq = dq.clamp(-max_step, max_step)
        q = torch.maximum(torch.minimum(q + dq, upper), lower)
    pos, _ = fk(q)
    return q, torch.linalg.norm(target_pos - pos, dim=1)


def initial_guess(rest_q, rest_pos, target_pos, lower, upper):
    """Rest posture with joint 1 (rotation about the base's vertical axis) turned so the tip's
    bearing from the base matches the target's."""
    turn = torch.atan2(target_pos[:, 1], target_pos[:, 0]) - torch.atan2(rest_pos[1], rest_pos[0])
    q = rest_q.clone()
    q[:, 0] = q[:, 0] + turn
    return torch.maximum(torch.minimum(q, upper), lower)


def world_to_goal_transforms(goal_to_world_base, goal_xy):
    """Per-episode 2D homogeneous world->goal transforms [n, 3, 3]: the base goal rotation,
    translated to each goal position. Used by the success check."""
    trans = goal_to_world_base.unsqueeze(0).repeat(goal_xy.shape[0], 1, 1)
    trans[:, 0:2, 2] = goal_xy
    return torch.linalg.inv(trans)


def _register():
    from mani_skill.envs.tasks.tabletop.push_t import PushTEnv
    from mani_skill.utils.registration import register_env
    from mani_skill.utils.structs.pose import Pose
    from transforms3d.euler import euler2quat

    @register_env(ENV_ID, max_episode_steps=100)
    class PushTVariableGoalEnv(PushTEnv):
        def __init__(self, *args, goal_quadrants=(1, 2, 3, 4), start_quadrants=(1, 2, 3, 4),
                     goal_half_width=GOAL_HALF_WIDTH, random_tip_start=True, **kwargs):
            self.goal_quadrants = _check_quadrants(goal_quadrants, "goal_quadrants")
            self.start_quadrants = _check_quadrants(start_quadrants, "start_quadrants")
            self.goal_half_width = float(goal_half_width)
            self.random_tip_start = bool(random_tip_start)
            super().__init__(*args, **kwargs)

        def _load_scene(self, options: dict):
            super()._load_scene(options)
            n, dev = self.num_envs, self.device
            # Base goal-to-world transform (rotation fixed), copied per environment.
            self._goal_trans_base = torch.linalg.inv(self.world_to_goal_trans)
            self.world_to_goal_trans = self.world_to_goal_trans.unsqueeze(0).repeat(n, 1, 1).clone()
            self.goal_xy = self.goal_offset.to(dev).unsqueeze(0).repeat(n, 1).clone()
            self.goal_quadrant = torch.zeros(n, dtype=torch.long, device=dev)
            self.tip_start_xy = torch.zeros(n, 2, device=dev)
            self.tip_start_quadrant = torch.zeros(n, dtype=torch.long, device=dev)
            self.tee_start_quadrant = torch.zeros(n, dtype=torch.long, device=dev)
            self.tip_ik_error = torch.zeros(n, device=dev)
            self.tip_ik_fallback = torch.zeros(n, dtype=torch.bool, device=dev)
            self._ik = None

        def _setup_ik(self):
            """Kinematic chain and the start lookup table, built once per process and device."""
            center = tuple(float(x) for x in self.goal_offset)
            key = (str(self.device), center, TIP_HALF_WIDTH, TIP_GRID)
            if key not in _IK_TABLES:
                _IK_TABLES[key] = _build_ik_table(self.agent, self.device, torch.tensor(center, device=self.device))
            self._ik = _IK_TABLES[key]

        def _initialize_episode(self, env_idx: torch.Tensor, options: dict):
            with torch.device(self.device):
                b = len(env_idx)
                self.table_scene.initialize(env_idx)  # robot rest pose + small joint noise, as PushT-v1
                center = self.goal_offset.to(self.device)

                # Goal: random position in an allowed quadrant, fixed rotation.
                offsets, quadrant = sample_goal_offsets(b, self.goal_quadrants, self.goal_half_width, device=self.device)
                goal_xy = center + offsets
                goal_xyz = torch.zeros((b, 3))
                goal_xyz[:, :2] = goal_xy
                goal_xyz[:, 2] = 1e-3
                self.goal_tee.set_pose(Pose.create_from_pq(p=goal_xyz, q=euler2quat(0, 0, self.goal_z_rot)))
                self.goal_xy[env_idx] = goal_xy
                self.goal_quadrant[env_idx] = quadrant
                self.world_to_goal_trans[env_idx] = world_to_goal_transforms(self._goal_trans_base, goal_xy)

                # Movable T: PushT-v1's original absolute spawn region, random rotation.
                tee_xyz = torch.zeros((b, 3))
                tee_xyz[:, 0] = center[0] + torch.rand(b) * self.tee_spawnbox_xlength + self.tee_spawnbox_xoffset
                tee_xyz[:, 1] = center[1] + torch.rand(b) * self.tee_spawnbox_ylength + self.tee_spawnbox_yoffset
                tee_xyz[:, 2] = 0.04 / 2 + 1e-3
                angle = torch.rand(b) * (2 * torch.pi)
                q = torch.zeros((b, 4))
                q[:, 0] = (angle / 2).cos()
                q[:, -1] = (angle / 2).sin()
                self.tee.set_pose(Pose.create_from_pq(p=tee_xyz, q=q))
                self.tee_start_quadrant[env_idx] = quadrant_of(tee_xyz[:, :2] - center)

                # Stick tip: random start away from the T, placed by IK (independent of the goal).
                if self.random_tip_start:
                    self._place_tip(env_idx, tee_xyz[:, :2] - center, center)
                else:
                    tip = self.ee_starting_pos2D[:2].to(self.device).expand(b, 2)
                    self.tip_start_xy[env_idx] = tip
                    self.tip_start_quadrant[env_idx] = quadrant_of(tip - center)
                    self.tip_ik_error[env_idx] = 0.0
                    self.tip_ik_fallback[env_idx] = False

                # End-effector marker, as in PushT-v1.
                xyz = torch.zeros((b, 3))
                xyz[:] = self.ee_starting_pos2D
                self.ee_goal_pos.set_pose(Pose.create_from_pq(p=xyz, q=euler2quat(0, np.pi / 2, 0)))

        def _place_tip(self, env_idx, tee_offset_xy, center):
            if self._ik is None:
                self._setup_ik()
            ik = self._ik
            b = len(env_idx)
            cell, _ = choose_tip_cells(tee_offset_xy, self.start_quadrants, ik["valid"], ik["grid"])
            fallback = ~ik["valid"][cell]
            # PushT-v1's rest posture if no reachable cell was found for an episode.
            q = torch.where(fallback.unsqueeze(1), ik["rest"].expand(b, -1), ik["q"][cell])
            self.agent.reset(q)
            tip = torch.where(fallback.unsqueeze(1), ik["rest_tip_xy"].expand(b, 2), ik["tip_xy"][cell])
            self.tip_start_xy[env_idx] = tip
            self.tip_start_quadrant[env_idx] = quadrant_of(tip - center)
            self.tip_ik_error[env_idx] = torch.where(fallback, torch.zeros(b, device=tip.device), ik["error"][cell])
            self.tip_ik_fallback[env_idx] = fallback

    return PushTVariableGoalEnv


_IK_TABLES = {}


def _build_ik_table(agent, device, center):
    """Solve IK once for every grid cell centre. Returns joint angles, achieved tip positions (world xy),
    position errors and a reachability mask, plus PushT-v1's rest posture and tip."""
    import pytorch_kinematics as pk

    with open(agent.urdf_path, "rb") as f:
        chain = pk.build_serial_chain_from_urdf(f.read(), end_link_name=agent.ee_link_name)
    chain = chain.to(device=device, dtype=torch.float32)
    lim = torch.as_tensor(chain.get_joint_limits(), dtype=torch.float32, device=device)
    rest = torch.as_tensor(agent.keyframes["rest"].qpos, dtype=torch.float32, device=device)
    joints = chain.get_joint_parameter_names()
    assert len(joints) == rest.numel(), (joints, rest.numel())
    base = torch.tensor(ROBOT_BASE_XYZ, device=device)

    def fk(q):
        m = chain.forward_kinematics(q).get_matrix()
        return m[:, :3, 3], m[:, :3, :3]

    rest_pos, rest_rot = fk(rest.unsqueeze(0))
    grid = tip_grid_offsets(TIP_HALF_WIDTH, TIP_GRID, device)
    k = grid.shape[0]
    target_pos = torch.zeros((k, 3), device=device)
    target_pos[:, :2] = center + grid - base[:2]
    target_pos[:, 2] = rest_pos[0, 2]
    q0 = initial_guess(rest.expand(k, -1), rest_pos[0], target_pos, lim[0], lim[1])
    q, err = dls_ik(fk, lambda q: chain.jacobian(q), q0, target_pos, rest_rot.expand(k, 3, 3), lim[0], lim[1])
    tip_xy = fk(q)[0][:, :2] + base[:2]
    valid = err <= IK_TOLERANCE
    return dict(grid=grid, q=q, tip_xy=tip_xy, error=err, valid=valid, rest=rest,
                rest_tip_xy=rest_pos[0, :2] + base[:2], valid_fraction=valid.float().mean().item())


def _check_quadrants(quadrants, name):
    quadrants = tuple(sorted({int(q) for q in quadrants}))
    if not quadrants or not set(quadrants) <= {1, 2, 3, 4}:
        raise ValueError(f"{name} must be a non-empty subset of 1-4, got {quadrants}")
    return quadrants


try:
    PushTVariableGoalEnv = _register()
except ImportError:  # ManiSkill not installed (offline tests use the helpers only)
    PushTVariableGoalEnv = None


In [ ]:
BC = WORK_ROOT / "pusht_state_bc.py"
for name in ("pusht_state_bc.py", "variable_goal_env.py"):
    shutil.copy2(WORK_ROOT / name, OUTPUT_ROOT / name)
run([TRAIN_PY, BC, "--help"])

## 4. Check the environment

`audit` records the observation contract (now with a varying goal). `check-env` runs two checks **before** any time goes into PPO:
- **Success check:** places a stand-in T exactly on each episode's goal and confirms the success test counts it, then shifts it
  5 cm and confirms it does not. Catches a success check still pointing at the old fixed goal.
- **Tip starts:** compares where the simulator actually put the stick tip with where inverse kinematics aimed it (must be
  within 2 cm), requires at most 5% of episodes to fall back to PushT-v1's start, and reports tip height, distance to the T,
  and start quadrants.

In [ ]:
run([TRAIN_PY, BC, "audit", "--env-id", ENV_ID, "--out", OUTPUT_ROOT])
contract = json.loads((OUTPUT_ROOT / "observation_contract.json").read_text())
assert contract["control_mode"] == "pd_joint_delta_pos" and contract["state_dim"] == 31, contract
run([TRAIN_PY, BC, "check-env", "--env-id", ENV_ID, "--num-envs", 256, "--out", OUTPUT_ROOT / "env_check.json"])

## 5. Train the goal-conditioned PPO expert

ManiSkill's `ppo.py`, unchanged, run with the variable-goal environment registered. Only evaluation lines are shown;
everything is in `notebook.log`. Checkpoints land in `ppo/runs/<name>/`. Set `PPO_CHECKPOINT` in the config cell to skip this.
`eval_success_once_mean` is ppo.py's own 16-episode evaluation, a rough progress signal.

In [ ]:
PPO_DIR = OUTPUT_ROOT / "ppo"
PPO_DIR.mkdir(exist_ok=True)
PPO_RUN = f"ppo_variable_goal_seed{PPO_SEED}"
if not PPO_CHECKPOINT:
    run([TRAIN_PY, BC, "ppo", "--script", PPO_SCRIPT,
         "--env-id", ENV_ID, "--control-mode", "pd_joint_delta_pos", "--exp-name", PPO_RUN, "--seed", PPO_SEED,
         "--num-envs", PPO_NUM_ENVS, "--num-steps", PPO_NUM_STEPS, "--num-minibatches", PPO_MINIBATCHES,
         "--update-epochs", PPO_EPOCHS, "--gamma", 0.99, "--total-timesteps", PPO_TIMESTEPS,
         "--num-eval-envs", 16, "--num-eval-steps", 100, "--eval-freq", PPO_EVAL_FREQ,
         "--save-model", "--no-track", "--no-capture-video"],
        show=r"eval_success|eval_return|SPS|Epoch: \d*00,|rror|Traceback", cwd=PPO_DIR)

## 6. Pick the PPO expert on validation seeds, per quadrant

Evaluates the final checkpoint and the latest periodic ones on 200 validation episodes, keeps the best
(success-once, then success-at-end), and stops if it is below `MIN_EXPERT_SUCCESS_ONCE`.

In [ ]:
def show_quadrants(name, res):
    row = "  ".join(f"{q} {v['success_once']:.0%}/{v['success_at_end']:.0%} (n={v['episodes']})"
                    for q, v in sorted(res.get("by_goal_quadrant", {}).items()))
    print(f"{name:<28} once {res['success_once']:.0%}  end {res['success_at_end']:.0%}   {row}")


if PPO_CHECKPOINT:
    candidates = [Path(PPO_CHECKPOINT)]
else:
    run_dir = PPO_DIR / "runs" / PPO_RUN
    periodic = sorted(run_dir.glob("ckpt_*.pt"), key=lambda p: int(p.stem.split("_")[1]))
    candidates = [run_dir / "final_ckpt.pt"] + periodic[-(PPO_CANDIDATES - 1):]
expert_evals = {}
for ckpt in candidates:
    out = OUTPUT_ROOT / f"ppo_val_{ckpt.stem}.json"
    run([TRAIN_PY, BC, "eval", "--env-id", ENV_ID, "--checkpoint", ckpt, "--ppo",
         "--episodes", EVAL_EPISODES, "--seed", VAL_SEED, "--out", out], show=r"^$")
    expert_evals[ckpt] = json.loads(out.read_text())
    show_quadrants(ckpt.name, expert_evals[ckpt])
EXPERT = max(expert_evals, key=lambda c: (expert_evals[c]["success_once"], expert_evals[c]["success_at_end"]))
TEACHERS = OUTPUT_ROOT / "teachers"
TEACHERS.mkdir(exist_ok=True)
shutil.copy2(EXPERT, TEACHERS / "ppo_expert.pt")
EXPERT_SOURCE = str(EXPERT)
print("Selected expert:", EXPERT)
if expert_evals[EXPERT]["success_once"] < MIN_EXPERT_SUCCESS_ONCE:
    raise RuntimeError(f"Expert success-once {expert_evals[EXPERT]['success_once']:.0%} is below "
                       f"{MIN_EXPERT_SUCCESS_ONCE:.0%}. Train PPO longer or check the environment before distilling BC.")
EXPERT = TEACHERS / "ppo_expert.pt"

## 7. Collect expert data

16,384 expert episodes (about 1.6M transitions) with action noise cycling 0, 0.1, 0.2, 0.3 per batch, plus 1,024
noise-free episodes on separate seeds for validation action error. Goals cover all four quadrants. Labels are always
the expert's noise-free action.

In [ ]:
DATA_DIR = OUTPUT_ROOT / "data"
EXPERT_DATA, VAL_DATA = DATA_DIR / "expert_rollouts.npz", DATA_DIR / "expert_val.npz"
run([TRAIN_PY, BC, "collect", "--env-id", ENV_ID, "--expert", EXPERT, "--driver", "expert",
     "--episodes", EXPERT_EPISODES, "--num-envs", NUM_ENVS, "--noise-std", *EXPERT_NOISE,
     "--seed-base", EXPERT_SEED_BASE, "--out", EXPERT_DATA])
run([TRAIN_PY, BC, "collect", "--env-id", ENV_ID, "--expert", EXPERT, "--driver", "expert",
     "--episodes", VAL_EPISODES, "--num-envs", NUM_ENVS, "--noise-std", 0.0,
     "--seed-base", VAL_SEED_BASE, "--out", VAL_DATA])

## 8. Smoke test

Short BC run to catch problems before the main training.

In [ ]:
run([TRAIN_PY, BC, "train", "--env-id", ENV_ID, "--train-data", EXPERT_DATA, "--val-data", VAL_DATA,
     "--out", OUTPUT_ROOT / "smoke", "--seed", SEED, "--iters", 2000, "--log-every", 500,
     "--eval-every", 1000, "--eval-episodes", 32, "--val-seed", VAL_SEED])

## 9. Train the BC teacher (no DAgger)

One training run on the expert rollouts. Logs validation action error every 1,000 iterations and closed-loop validation
success every `EVAL_EVERY`. `best.pt` has the highest validation success-at-end (ties broken by success-once).

In [ ]:
RUN_DIR = OUTPUT_ROOT / "runs" / TEACHER_NAME
config = dict(teacher=TEACHER_NAME, env_id=ENV_ID, seed=SEED, iters=ITERS, batch_size=BATCH_SIZE, lr=LR,
              eval_every=EVAL_EVERY, eval_episodes=EVAL_EPISODES, val_seed=VAL_SEED, test_seed=TEST_SEED,
              expert=str(EXPERT), expert_episodes=EXPERT_EPISODES, expert_noise=EXPERT_NOISE, dagger=False,
              maniskill_commit=MANISKILL_COMMIT, architecture="MLP 2x256 ReLU, tanh output")
RUN_DIR.mkdir(parents=True, exist_ok=False)
(RUN_DIR / "run_config.json").write_text(json.dumps(config, indent=2))
run([TRAIN_PY, BC, "train", "--env-id", ENV_ID, "--train-data", EXPERT_DATA, "--val-data", VAL_DATA, "--out", RUN_DIR,
     "--seed", SEED, "--iters", ITERS, "--batch-size", BATCH_SIZE, "--lr", LR, "--eval-every", EVAL_EVERY,
     "--eval-episodes", EVAL_EPISODES, "--val-seed", VAL_SEED], show=r"iter +\d*[05]000 |Training on|Done")
shutil.copy2(RUN_DIR / "best.pt", TEACHERS / "bc_teacher.pt")

## 10. Training curves

In [ ]:
import csv
import matplotlib.pyplot as plt

rows = list(csv.DictReader((RUN_DIR / "train_log.csv").open()))
it = [int(r["iter"]) for r in rows]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
ax1.plot(it, [float(r["train_mse"]) for r in rows], label="train")
ax1.plot(it, [float(r["val_mse"]) for r in rows], label="validation")
ax1.set(xlabel="iteration", ylabel="action MSE", yscale="log", title="Action error")
ax1.legend()
ev = [r for r in rows if r["success_at_end"]]
ax2.plot([int(r["iter"]) for r in ev], [float(r["success_once"]) for r in ev], "o-", label="success once")
ax2.plot([int(r["iter"]) for r in ev], [float(r["success_at_end"]) for r in ev], "s-", label="success at end")
ax2.set(xlabel="iteration", ylabel="rate", ylim=(0, 1.02), title="Closed-loop validation")
ax2.legend()
fig.tight_layout()
fig.savefig(RUN_DIR / "training_curves.png", dpi=120)
plt.show()

## 11. Final test on fresh seeds, per quadrant

Both teachers on the same 200 test episodes (seed 5001, never used for selection), broken down by goal quadrant.
Also saves a few BC videos.

In [ ]:
tests = {}
for name, ckpt, flags in (("ppo_expert", EXPERT, ["--ppo"]), ("bc_teacher", TEACHERS / "bc_teacher.pt", [])):
    out = TEACHERS / f"{name}_test_eval.json"
    run([TRAIN_PY, BC, "eval", "--env-id", ENV_ID, "--checkpoint", ckpt, *flags, "--episodes", EVAL_EPISODES,
         "--seed", TEST_SEED, "--out", out], show=r"^$")
    tests[name] = json.loads(out.read_text())
    show_quadrants(name, tests[name])
run([TRAIN_PY, BC, "eval", "--env-id", ENV_ID, "--checkpoint", TEACHERS / "bc_teacher.pt", "--episodes", 4,
     "--seed", TEST_SEED, "--video-dir", RUN_DIR / "videos"], show=r"^$")

summary = {"env_id": ENV_ID, "expert_checkpoint_source": EXPERT_SOURCE,
           "expert_validation": {str(k): v for k, v in expert_evals.items()},
           "test": tests, "teachers": sorted(p.name for p in TEACHERS.glob("*.pt"))}
(TEACHERS / "summary.json").write_text(json.dumps(summary, indent=2))
print("Teachers saved in", TEACHERS)

## 12. Watch a rollout

In [ ]:
from IPython.display import Video, display
videos = sorted((RUN_DIR / "videos").glob("*.mp4"))
print(len(videos), "videos")
if videos:
    display(Video(str(videos[0]), embed=True, width=480))